# 01. Agent evals and context

Priya asks, “Which region led?” The assistant replies, “Which month should I check?” Now suppose it also looked up January before it asked. How would you catch that?

Run the cells from the top. The sample runs are dictionaries, and the code uses only Python's standard library. You can start here any time. No API or model is needed.

## What to check

An **eval** compares an agent run with an expectation you wrote ahead of time. A **trace** records what happened during the run. It can show which tool was called, what arguments it received, and what it returned.

| Check | Example |
|---|---|
| Final answer | The March reply names North and $1,200. |
| Evidence | The amount in the reply agrees with the tool result. |
| Tool and arguments | A March question calls the sales tool with March. |
| Clarification | A question with no month asks for the month and makes zero calls. |
| Missing data and failures | A report with no April rows gets a no-data reply. A failed lookup gets an error or fallback. |
| Side effects | A request for an email draft leaves the outbox empty. |
| Speed and cost | Record latency, tokens, or tool-call count when they matter to the app. |

Use Python for exact checks such as an ID or a count. For a quality check like “Is this answer clear?”, write a short set of grading rules and compare a few grades with your own judgment. [Agent evals](https://www.anthropic.com/research/demystifying-evals-for-ai-agents)

## Grade two runs

Read both sample replies. Predict which run should pass. The requirement is to ask for a month with zero tool calls.

In [ ]:
question = "Which region led?"
expected_reply = "Which month should I check?"

sample_runs = [
    {
        "name": "Looked up January first",
        "answer": "Which month should I check?",
        "tool_calls": [{"name": "sales_by_region", "month": "January"}],
    },
    {
        "name": "Asked before looking up sales",
        "answer": "Which month should I check?",
        "tool_calls": [],
    },
]

print("User question:", question)
for run in sample_runs:
    # Grade what the user saw and what the agent did separately.
    answer_ok = run["answer"] == expected_reply
    tools_ok = len(run["tool_calls"]) == 0
    print(run["name"])
    print("  Answer:", answer_ok, "| Tool use:", tools_ok)
    print("  Case:", "PASS" if answer_ok and tools_ok else "FAIL")

Both replies pass the answer check. Only the second run passes the tool check. The first looked up January before asking for a month, so the overall case fails.

For an agent you build, start with a normal question, missing input, a boundary case, and a tool failure. Save the expected behavior with each question. Rerun the same set after a change to the model, prompt, or tools.

**Try it.** Add a sample run with an empty tool-call list and a reply that guesses North. Which check fails?

## Choose what the model sees

**Context engineering** is deciding what information to send to the model for a particular step. That can include instructions, the current question, a few earlier messages, and results from tools.

For a March question, Python can filter and total the rows before sending the result. Run the cell and compare the two payloads. [Context engineering](https://www.anthropic.com/research/effective-context-engineering-for-ai-agents)

In [ ]:
import json

# The full file includes sales from another month.
full_report = [
    {"month": "March", "region": "North", "sales": 700},
    {"month": "March", "region": "South", "sales": 900},
    {"month": "March", "region": "East", "sales": 600},
    {"month": "March", "region": "North", "sales": 500},
    {"month": "February", "region": "South", "sales": 1800},
]

# The sales tool can filter and add the rows before replying to the model.
march_totals = {}
for row in full_report:
    if row["month"] == "March":
        region = row["region"]
        march_totals[region] = march_totals.get(region, 0) + row["sales"]

tool_result = {"month": "March", "totals": march_totals}

for label, data in [("Full report", full_report), ("Tool result", tool_result)]:
    payload = json.dumps(data)
    print(label, f"({len(payload)} characters)")
    print(payload)
    print()

The smaller payload keeps the month and the three totals. The lengths printed here are characters. Token counts depend on the model's tokenizer.

**Try it.** Add another March sale for South worth $300. North and South should tie at $1,200. Check that the short payload still contains both totals.

To inspect live tool calls, open [Build and evaluate a sales agent](02_sales_agent_design_and_evals.ipynb).